# WAN 2.2 14B — ComfyUI

WAN 2.2 14B ailesi: I2V, Animate Character Swap, Pose Control VACE.

**Pipeline:** `Görsel/Prompt → WAN 2.2 14B → Video`

## Ön Hazırlık
- Cloudflare Dashboard: `comfy.ersamely.com` → `localhost:8188`
- Colab Secrets: `CF_TUNNEL_TOKEN`, `HF_TOKEN`

## Kullanım
A: Kurulum + Node'lar → B: Model indir → C: Başlat + Tunnel → Browser: `comfy.ersamely.com`

---
# A) Kurulum + Custom Node'lar

In [ ]:
import os
import subprocess

import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU bulunamadı!')
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'\u2705 GPU: {gpu_name} ({gpu_mem:.1f} GB)')

os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('\u2705 HF_TOKEN')
except Exception:
    print('\u26a0\ufe0f HF_TOKEN yok')

# ComfyUI
COMFY_DIR = '/content/ComfyUI'
CUSTOM_NODES = f'{COMFY_DIR}/custom_nodes'

if not os.path.exists(COMFY_DIR):
    print('\U0001f4e6 ComfyUI...')
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git {COMFY_DIR}
    !pip install -q -r {COMFY_DIR}/requirements.txt
else:
    print('\u2705 ComfyUI mevcut')

# ─── Custom Node'lar ───
NODES = {
    # Video
    'ComfyUI-VideoHelperSuite': 'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git',
    'ComfyUI-Frame-Interpolation': 'https://github.com/Fannovel16/ComfyUI-Frame-Interpolation.git',
    'ComfyUI-VideoUpscale_WithModel': 'https://github.com/ShmuelRonen/ComfyUI-VideoUpscale_WithModel.git',
    # Utility
    'ComfyUI-Manager': 'https://github.com/ltdrdata/ComfyUI-Manager.git',
    'ComfyUI-Impact-Pack': 'https://github.com/ltdrdata/ComfyUI-Impact-Pack.git',
    'rgthree-comfy': 'https://github.com/rgthree/rgthree-comfy.git',
    # WAN Animate + Pose
    'ComfyUI-WanVideoWrapper': 'https://github.com/kijai/ComfyUI-WanVideoWrapper.git',
    'ComfyUI-KJNodes': 'https://github.com/kijai/ComfyUI-KJNodes.git',
    'ComfyUI-WanAnimatePreprocess': 'https://github.com/kijai/ComfyUI-WanAnimatePreprocess.git',
    'ComfyUI-segment-anything-2': 'https://github.com/kijai/ComfyUI-segment-anything-2.git',
    # Pose Control
    'comfyui_controlnet_aux': 'https://github.com/Fannovel16/comfyui_controlnet_aux.git',
    'ComfyUI-Florence2': 'https://github.com/kijai/ComfyUI-Florence2.git',
}

for name, url in NODES.items():
    node_dir = f'{CUSTOM_NODES}/{name}'
    if not os.path.exists(node_dir):
        print(f'  \u2193 {name}')
        !git clone --depth 1 {url} {node_dir}
        req_file = f'{node_dir}/requirements.txt'
        if os.path.exists(req_file):
            !pip install -q -r {req_file}
    else:
        print(f'  \u2713 {name}')

# Cloudflare Tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print('\n\u2705 Kurulum tamam')

---
# B) WAN 2.2 Model İndir

In [ ]:
import os
import glob

MODELS_DIR = f'{COMFY_DIR}/models'

# ╔══════════════════════════════════════════════════════════════╗
# ║  WORKFLOW SEÇ — değiştirip B'yi tekrar çalıştır            ║
# ║  Eski workflow modelleri otomatik silinir, ortak kalanlar   ║
# ╚══════════════════════════════════════════════════════════════╝
WORKFLOW = 'animate_character_swap'  # 'animate_character_swap' | 'pose_control_vace'


def hf_download(repo, filename, dest_dir):
    basename = filename.split('/')[-1]
    dest = f'{dest_dir}/{basename}'
    if os.path.exists(dest):
        print(f'  \u2713 {basename} (mevcut)')
        return
    print(f'  \u2193 {basename}...')
    os.makedirs(dest_dir, exist_ok=True)
    from huggingface_hub import hf_hub_download
    try:
        path = hf_hub_download(repo_id=repo, filename=filename, local_dir='/content/hf_cache')
        import shutil
        shutil.move(path, dest)
        print(f'  \u2705 {basename}')
    except Exception as e:
        print(f'  \u274c Başarısız: {e}')


def remove_if_exists(path):
    if os.path.exists(path):
        os.remove(path)
        print(f'  \U0001f5d1 Silindi: {os.path.basename(path)}')


# ─── Ortak modeller (her workflow'da lazım, silinmez) ───
print('\U0001f4e5 Ortak modeller:')
hf_download('Comfy-Org/Wan_2.2_ComfyUI_Repackaged', 'split_files/vae/wan_2.1_vae.safetensors', f'{MODELS_DIR}/vae')
hf_download('Comfy-Org/Wan_2.1_ComfyUI_repackaged', 'split_files/clip_vision/clip_vision_h.safetensors', f'{MODELS_DIR}/clip_vision')
hf_download('Comfy-Org/Wan_2.1_ComfyUI_repackaged', 'split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors', f'{MODELS_DIR}/text_encoders')

# ─── Workflow'a özel modeller ───
if WORKFLOW == 'i2v':
    print('\n\U0001f3ac Workflow: I2V (Image-to-Video)')
    # Önce eski modelleri sil
    remove_if_exists(f'{MODELS_DIR}/diffusion_models/Wan2_2-Animate-14B_fp8_e4m3fn_scaled_KJ.safetensors')
    remove_if_exists(f'{MODELS_DIR}/loras/WanAnimate_relight_lora_fp16.safetensors')
    remove_if_exists(f'{MODELS_DIR}/loras/lightx2v_I2V_14B_480p_cfg_step_distill_rank64_bf16.safetensors')
    remove_if_exists(f'{MODELS_DIR}/diffusion_models/wan2.2_t2v_14B_fp8_scaled.safetensors')
    remove_if_exists(f'{MODELS_DIR}/loras/lightx2v_T2V_14B_480p_cfg_step_distill_rank64_bf16.safetensors')
    # Sonra indir
    hf_download('Comfy-Org/Wan_2.2_ComfyUI_Repackaged', 'split_files/diffusion_models/wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors', f'{MODELS_DIR}/diffusion_models')
    hf_download('Comfy-Org/Wan_2.2_ComfyUI_Repackaged', 'split_files/diffusion_models/wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors', f'{MODELS_DIR}/diffusion_models')
    hf_download('Comfy-Org/Wan_2.2_ComfyUI_Repackaged', 'split_files/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors', f'{MODELS_DIR}/loras')
    hf_download('Comfy-Org/Wan_2.2_ComfyUI_Repackaged', 'split_files/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors', f'{MODELS_DIR}/loras')

elif WORKFLOW == 'animate_character_swap':
    print('\n\U0001f3ac Workflow: Animate Character Swap')
    # Önce eski modelleri sil
    remove_if_exists(f'{MODELS_DIR}/diffusion_models/wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors')
    remove_if_exists(f'{MODELS_DIR}/diffusion_models/wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors')
    remove_if_exists(f'{MODELS_DIR}/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors')
    remove_if_exists(f'{MODELS_DIR}/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors')
    remove_if_exists(f'{MODELS_DIR}/diffusion_models/wan2.2_t2v_14B_fp8_scaled.safetensors')
    remove_if_exists(f'{MODELS_DIR}/loras/lightx2v_T2V_14B_480p_cfg_step_distill_rank64_bf16.safetensors')
    # Sonra indir
    hf_download('Kijai/WanVideo_comfy_fp8_scaled', 'Wan22Animate/Wan2_2-Animate-14B_fp8_e4m3fn_scaled_KJ.safetensors', f'{MODELS_DIR}/diffusion_models')
    hf_download('Kijai/WanVideo_comfy', 'LoRAs/Wan22_relight/WanAnimate_relight_lora_fp16.safetensors', f'{MODELS_DIR}/loras')
    hf_download('Kijai/WanVideo_comfy', 'Lightx2v/lightx2v_I2V_14B_480p_cfg_step_distill_rank64_bf16.safetensors', f'{MODELS_DIR}/loras')

elif WORKFLOW == 'pose_control_vace':
    print('\n\U0001f3ac Workflow: Pose Control VACE')
    # Önce eski modelleri sil
    remove_if_exists(f'{MODELS_DIR}/diffusion_models/wan2.2_i2v_low_noise_14B_fp8_scaled.safetensors')
    remove_if_exists(f'{MODELS_DIR}/diffusion_models/wan2.2_i2v_high_noise_14B_fp8_scaled.safetensors')
    remove_if_exists(f'{MODELS_DIR}/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors')
    remove_if_exists(f'{MODELS_DIR}/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors')
    remove_if_exists(f'{MODELS_DIR}/diffusion_models/Wan2_2-Animate-14B_fp8_e4m3fn_scaled_KJ.safetensors')
    remove_if_exists(f'{MODELS_DIR}/loras/WanAnimate_relight_lora_fp16.safetensors')
    remove_if_exists(f'{MODELS_DIR}/loras/lightx2v_I2V_14B_480p_cfg_step_distill_rank64_bf16.safetensors')
    # Sonra indir
    hf_download('Comfy-Org/Wan_2.2_ComfyUI_Repackaged', 'split_files/diffusion_models/wan2.2_t2v_14B_fp8_scaled.safetensors', f'{MODELS_DIR}/diffusion_models')
    hf_download('Kijai/WanVideo_comfy', 'Lightx2v/lightx2v_T2V_14B_480p_cfg_step_distill_rank64_bf16.safetensors', f'{MODELS_DIR}/loras')

print('\n\u2705 Model indirme tamam')

---
# C) ComfyUI Başlat

`USE_CLOUDFLARE = False` (default): Colab proxy — hızlı, URL değişir
`USE_CLOUDFLARE = True`: Cloudflare tunnel — yavaş ama sabit URL

In [ ]:
import subprocess
import time

import requests
from google.colab import userdata, output

# ╔══════════════════════════════════════════════════╗
# ║  TUNNEL SEÇ                                    ║
# ╚══════════════════════════════════════════════════╝
USE_CLOUDFLARE = False  # False=Colab proxy (hızlı) | True=Cloudflare (sabit URL)

PORT = 8188

# Önceki process'leri kapat
subprocess.run(['pkill', '-f', 'main.py'], capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(2)

# ComfyUI başlat
log_file = open('/content/comfyui.log', 'w')
comfy_proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--port', str(PORT), '--gpu-only', '--enable-cors-header', '*'],
    cwd=COMFY_DIR,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    stdin=subprocess.DEVNULL,
)
print(f'\U0001f680 ComfyUI başlatıldı (PID: {comfy_proc.pid})')

# Hazır olmasını bekle
t0 = time.time()
ready = False
while time.time() - t0 < 120:
    try:
        if requests.get(f'http://localhost:{PORT}/system_stats', timeout=3).status_code == 200:
            ready = True
            break
    except requests.RequestException:
        pass
    if comfy_proc.poll() is not None:
        print('\u274c ComfyUI çöktü!')
        log_file.close()
        with open('/content/comfyui.log') as f:
            print(f.read()[-500:])
        break
    time.sleep(3)

if ready:
    print(f'\u2705 ComfyUI hazır ({int(time.time()-t0)}s)')

    if USE_CLOUDFLARE:
        token = userdata.get('CF_TUNNEL_TOKEN')
        cf_log = open('/content/cloudflared.log', 'w')
        cf_proc = subprocess.Popen(
            ['cloudflared', 'tunnel', '--no-autoupdate', 'run', '--token', token],
            stdout=cf_log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
        )
        time.sleep(5)
        print(f'\U0001f310 Cloudflare: https://comfyui.ersamely.com')
    else:
        print(f'\U0001f310 Colab Proxy:')
        output.serve_kernel_port_as_window(PORT, path='/')
else:
    print('\u274c Timeout!')

In [ ]:
import time
from datetime import datetime, timezone

import requests

print('ComfyUI canlı tutma. Durdurmak için interrupt et.\n')
while True:
    try:
        local_ok = requests.get(f'http://localhost:{PORT}/system_stats', timeout=5).status_code == 200
    except requests.RequestException:
        local_ok = False

    comfy_alive = comfy_proc.poll() is None
    now = datetime.now(timezone.utc).strftime('%H:%M:%S UTC')
    c = '\u2705' if (local_ok and comfy_alive) else '\u274c'
    print(f'{now} | ComfyUI: {c}')
    time.sleep(30)